<center><img src="image.png" width=500></center>
<p>

Has empezado recientemente como Data Engineer en una empresa del sector energético. Antes, los analistas de otros equipos tenían que recuperar y limpiar los datos manualmente cada trimestre para entender los cambios en las ventas y la capacidad de distintos tipos de energía. Este proceso solía llevar varios días y era algo que la mayoría de analistas temía. Tu trabajo es automatizarlo creando un pipeline de datos. Escribirás este pipeline para extraer datos cada mes, lo que ayudará a obtener insights más rápido y a liberar tiempo para tus usuarios de datos.

Lo lograrás usando la biblioteca `pandas` y sus potentes funcionalidades de análisis. Trabajarás con dos archivos en bruto: `electricity_sales.csv` y `electricity_capability_nested.json`. 
    
A continuación encontrarás un diccionario de datos del conjunto `electricity_sales.csv`, que transformarás en un momento. ¡Éxitos!

| Field | Data Type |
| :---- | :-------: |
| period  | `str`        |
| stateid | `str` |
| stateDescription | `str` |
| sectorid | `str` |
| sectorName | `str` |
| price | `float` |
| price-units | `str` |

In [14]:
import pandas as pd
import json

In [15]:
def extract_tabular_data(file_path: str):
    """Extract data from a tabular file_format, with pandas."""
    
    if file_path.endswith(".csv"):
        return pd.read_csv(file_path)
    
    elif file_path.endswith(".parquet"):
        return pd.read_parquet(file_path)
    
    else:
        raise Exception("Warning: Invalid file extension. Please try with .csv or .parquet!")
    

In [16]:
def extract_json_data(file_path):
    """Extract and flatten data from a JSON file."""
    
    with open(file_path, "r") as file:
        raw_json = json.load(file)
    
    return pd.json_normalize(raw_json)

In [17]:
def transform_electricity_sales_data(raw_data: pd.DataFrame):
    """
    Transform electricity sales to find the total amount of electricity sold
    in the residential and transportation sectors.
    """
    
    raw_data.dropna(subset=["price"], inplace=True)
    
    cleaned_data = raw_data.loc[
        raw_data["sectorName"].isin(["residential", "transportation"]),
        :
    ]
    
    cleaned_data["year"] = cleaned_data["period"].str[:4]
    cleaned_data["month"] = cleaned_data["period"].str[-2:]
    
    cleaned_data = cleaned_data.loc[
        :,
        ["year", "month", "stateid", "price", "price-units"]
    ]
    
    return cleaned_data

In [18]:
def load(dataframe: pd.DataFrame, file_path: str):
    """Load a DataFrame to a file in either CSV or Parquet format."""
    
    if file_path.endswith(".csv"):
        dataframe.to_csv(file_path, index=False)
    
    elif file_path.endswith(".parquet"):
        dataframe.to_parquet(file_path, index=False)
    
    else:
        raise Exception(f"Warning: {file_path} is not a valid file type. Please try again!_")

In [19]:
raw_electricity_capability_df = extract_json_data("electricity_capability_nested.json")
raw_electricity_sales_df = extract_tabular_data("electricity_sales.csv")

cleaned_electricity_sales_df = transform_electricity_sales_data(raw_electricity_sales_df)

load(raw_electricity_capability_df, "loaded__electricity_capability.parquet")
load(cleaned_electricity_sales_df, "loaded__electricity_sales.csv")